# FEMBA Phase 1: SSL Pre-Training (6 Datasets)

## Obiettivo
In questo notebook eseguiamo il pre-addestramento **Self-Supervised** dell'encoder **FEMBA** (2025) sui 6 dataset non etichettati.
Questo garantisce un confronto alla pari con Corponi e BioMamba, usando lo stesso identico set di dati di partenza.

In [ ]:
# 1. Setup Ambiente e Download FEMBA
import os, sys, shutil
!pip install -q --upgrade pip
!pip install -q packaging setuptools triton einops biosppy peakutils

print("Installazione causal-conv1d...")
!pip install -q causal-conv1d>=1.4.0 --no-build-isolation

print("Installazione mamba-ssm...")
%env MAX_JOBS=1
!pip install -q mamba-ssm --no-build-isolation

print("Installazione dipendenze generali...")
!pip install -q flirt ruamel.yaml hrv-analysis neurokit2 distinctipy joypy pytorch-lightning torchmetrics biopsykit

FEMBA_REPO = '/kaggle/working/FEMBA_REPO'
if os.path.exists(FEMBA_REPO): shutil.rmtree(FEMBA_REPO)
print("Cloning FEMBA Repository...")
!git clone --depth 1 https://github.com/pulp-bio/biofoundation.git {FEMBA_REPO}

WORK_DIR = '/kaggle/working/e4selflearning'
if not os.path.exists(WORK_DIR):
    !git clone https://github.com/april-tools/e4selflearning.git {WORK_DIR}

import os
print("✅ Setup completato. Riavvio kernel...")
os._exit(0)

In [ ]:
# 2. Ingestion, Patch & Segmentazione
import os, sys, glob, shutil, pickle, torch
WORK_DIR = '/kaggle/working/e4selflearning'
if WORK_DIR not in sys.path: sys.path.insert(0, f"{WORK_DIR}/src")
os.chdir(WORK_DIR)

def apply_patch(file_path, old_str, new_str):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f: c = f.read()
        if old_str in c:
            c = c.replace(old_str, new_str)
            with open(file_path, 'w') as f: f.write(c)
            print(f"✅ Patch applicata a {file_path}")

apply_patch('segment.py', 'newshape=', 'shape=')
apply_patch('src/timebase/data/preprocessing.py', 'newshape=', 'shape=')

TARGET_DIR = '/kaggle/working/e4selflearning/data/preprocessed/unsegmented_unlabelled'
if not os.path.exists(TARGET_DIR): os.makedirs(TARGET_DIR, exist_ok=True)

meta_files = glob.glob('/kaggle/input/**/metadata.pkl', recursive=True)
if not meta_files: raise FileNotFoundError("metadata.pkl non trovato!")

src_dir = os.path.dirname(meta_files[0])
with open(meta_files[0], 'rb') as f: meta = pickle.load(f)

print("Scansione profonda del dataset...")
all_h5 = glob.glob(os.path.join(src_dir, '**/channels.h5'), recursive=True)
new_sessions = {}
for h5_path in all_h5:
    session_dir = os.path.dirname(h5_path)
    session_id = os.path.basename(session_dir)
    dest_dir = os.path.join(TARGET_DIR, session_id)
    if not os.path.exists(dest_dir): os.symlink(session_dir, dest_dir)
    found = False
    for old_id in meta['sessions_info'].keys():
        if old_id.endswith(session_id):
            new_sessions[session_id] = meta['sessions_info'][old_id]
            found = True; break
    if not found: new_sessions[session_id] = list(meta['sessions_info'].values())[0] 

meta['sessions_info'] = new_sessions
with open(os.path.join(TARGET_DIR, 'metadata.pkl'), 'wb') as f: pickle.dump(meta, f)

print(f"✅ Ingestion completata: {len(new_sessions)} sessioni mappate.")
print("Inizio Segmentazione...")
!PYTHONPATH=src python segment.py --data_dir {TARGET_DIR} --output_dir data/preprocessed/sl640_ss640_unlabelled --segment_length 640 --segmentation_mode 1 --step_size 640 --num_workers 2

# Fix per static.py (Collezioni)
static_p = 'src/timebase/data/static.py'
if os.path.exists(static_p):
    with open(static_p, 'r') as f: c = f.read()
    if 'class CollectionsDict(dict):' not in c:
        fb = '\nclass CollectionsDict(dict):\n    def __getitem__(self, key):\n        return super().__getitem__(key) if key in self else 0\n'
        c = c.replace('COLLECTIONS_DICT = {', fb + 'COLLECTIONS_DICT = CollectionsDict({')
        c = c.replace('}\n\nCOLLECTIONS_COLOR_DICT', '})\n\nCOLLECTIONS_COLOR_DICT')
        with open(static_p, 'w') as f: f.write(c)
        print("✅ Patch applicata a static.py (KeyError Fix)")

In [ ]:
# 3. Architettura FEMBA per SSL (Fix Dimensioni)
import os, sys, torch, torch.nn as nn, torch.nn.functional as F
from einops import rearrange

FEMBA_REPO = '/kaggle/working/FEMBA_REPO'
if FEMBA_REPO not in sys.path: sys.path.append(FEMBA_REPO)
from models.FEMBA import FEMBA

class FEMBAReconstructor(nn.Module):
    def __init__(self, femba_model):
        super().__init__()
        self.encoder = femba_model
    def forward(self, x, mask):
        # FEMBA expects (B, C, T) internally. It will do its own unsqueeze(1).
        recon, orig = self.encoder(x, mask)
        return recon.squeeze(1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_model = FEMBA(seq_length=640, num_channels=6, num_classes=0, embed_dim=128, num_blocks=4).to(device)
model = FEMBAReconstructor(base_model).to(device)
print("✅ Architettura FEMBA Phase 1 pronta.")

In [ ]:
# 4. Training Loop (20 epochs)
from timebase.data.reader import get_datasets
class Args: dataset='data/preprocessed/sl640_ss640_unlabelled'; e4selflearning=True; batch_size=32; num_workers=2; pretext_task='masked_prediction'; masking_ratio=0.15; lm=3; scaling_mode=2; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); seed=42; filter_collections=None; include_hr=False; reuse_stats=False; exclude_anomalies=False; plot_mode=0; task_mode=0; use_wandb=False; unlabelled_data_resampling_percentage=1.0; verbose=0; split_mode=0; overwrite_masks=False

args = Args(); loader, _, _ = get_datasets(args, summary=None)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4); criterion = nn.MSELoss()

def dict_to_tensor(d, dev): 
    tensors = [F.interpolate(d[k].float().to(dev).unsqueeze(1), size=640, mode='linear').squeeze(1) for k in sorted(d.keys())]
    return torch.stack(tensors, dim=1)

print("Starting FEMBA SSL Pre-training (20 epochs)...")
for epoch in range(1, 21):
    model.train(); total_loss = 0.0
    for batch in loader:
        orig, corr, mask = dict_to_tensor(batch['original'], device), dict_to_tensor(batch['corrupted'], device), dict_to_tensor(batch['mask'], device).bool()
        optimizer.zero_grad()
        # Passiamo orig e mask al modello
        recon = model(orig, mask)
        loss = torch.sqrt(criterion(recon[mask], orig[mask]))
        loss.backward(); optimizer.step(); total_loss += loss.item()
    print(f"Epoch {epoch:2d}/20 | Loss: {total_loss/len(loader):.4f}")

torch.save(base_model.state_dict(), '/kaggle/working/femba_encoder_pretrained.pt')
print("FEMBA Pre-trained weights saved!")
